In [42]:
import os, glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import Input, Embedding, Flatten, LSTM, GRU, Concatenate, Dense, Dropout
from tensorflow.keras.models import Model

In [43]:
WINDOW = 60
FEATURES = ['open', 'high', 'low', 'close', 'per_change', 'traded_quantity',
            'ma_7', 'ma_21', 'volatility', 'price_range']
TARGET = 'close'
MIN_YEAR = 2015

# Column-name candidates, in priority order. Update these lists if your
# CSV uses something not listed here.
DATE_CANDIDATES = ["Date", "date", "published_date", "Published Date"]
SYMBOL_CANDIDATES = ["Symbol", "symbol", "Company", "company", "ticker", "Ticker"]

In [44]:
def _resolve_column(df, candidates, role):
    """Find which of the candidate column names is actually present, and
    return it. Raises a clear error (listing real columns) if none match,
    instead of a confusing KeyError deep inside pandas."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not find a {role} column. Tried {candidates}, "
        f"but the DataFrame only has: {df.columns.tolist()}. "
        f"Add the real column name to the *_CANDIDATES list at the top of the script."
    )


def normalize_columns(df, filepath=None):
    """Rename whichever date/symbol columns actually exist to the
    canonical 'Date' / 'Symbol' names the rest of the pipeline expects.

    If there's no symbol/ticker column at all (common when each CSV holds
    a single company's history), the symbol is derived from the filename
    instead, e.g. 'data/NABIL.csv' -> Symbol = 'NABIL'."""
    df = df.copy()

    date_col = _resolve_column(df, DATE_CANDIDATES, "date")
    rename_map = {}
    if date_col != "Date":
        rename_map[date_col] = "Date"
    df = df.rename(columns=rename_map)
    df["Date"] = pd.to_datetime(df["Date"])

    symbol_col = None
    for c in SYMBOL_CANDIDATES:
        if c in df.columns:
            symbol_col = c
            break

    if symbol_col is not None:
        if symbol_col != "Symbol":
            df = df.rename(columns={symbol_col: "Symbol"})
    else:
        if filepath is None:
            raise KeyError(
                f"No symbol column found (tried {SYMBOL_CANDIDATES}) and no "
                f"filepath given to derive one from. Columns present: "
                f"{df.columns.tolist()}"
            )
        # No symbol column in the file itself -> assume this CSV is a
        # single company's history and name it after the file.
        inferred_symbol = os.path.splitext(os.path.basename(filepath))[0]
        df["Symbol"] = inferred_symbol
        print(f"No symbol column found in {filepath}; "
              f"using filename as Symbol = '{inferred_symbol}'.")

    return df

In [45]:
def load_stock_data(filepath):
    df = pd.read_csv(filepath)
    df = normalize_columns(df, filepath=filepath)
    df.sort_values("Date", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

In [46]:
def load_and_clean(df):
    df = df.copy()  # never mutate the caller's DataFrame in place
    df = df.sort_values("Date").reset_index(drop=True)
    df = df.drop_duplicates(subset=["Symbol", "Date"])
    df = df[df["Date"].dt.year >= MIN_YEAR]
    df.dropna(subset=["close"], inplace=True)

    if "traded_amount" in df.columns:
        df.drop(columns=["traded_amount"], inplace=True)
    if "status" in df.columns:
        df.drop(columns=["status"], inplace=True)

    df["per_change"] = df["per_change"].fillna(0.0)
    for col in ["open", "high", "low", "traded_quantity"]:
        df[col] = df[col].ffill().bfill()

    df["ma_7"] = df["close"].rolling(7, min_periods=1).mean()
    df["ma_21"] = df["close"].rolling(21, min_periods=1).mean()
    df["volatility"] = df["close"].rolling(7, min_periods=1).std().fillna(0)
    df["price_range"] = df["high"] - df["low"]
    return df

In [47]:
def encode_categoricals(df):
    df = df.copy()
    symbols = sorted(df["Symbol"].unique())
    symbol_to_id = {s: i for i, s in enumerate(symbols)}
    df["SymbolID"] = df["Symbol"].map(symbol_to_id)
    return df, len(symbols), symbol_to_id

In [48]:
def check_symbol_coverage(df, window=WINDOW, train_frac=0.7):
    summary = df.groupby("Symbol")["Date"].agg(["min", "max", "count"])
    summary["train_rows_est"] = (summary["count"] * train_frac).astype(int)
    summary["enough_for_one_window"] = summary["train_rows_est"] > window
    print(summary)
    too_short = summary[~summary["enough_for_one_window"]]
    if len(too_short) > 0:
        print("\nWARNING — these symbols won't produce training sequences "
              f"with WINDOW={window}:")
        print(too_short.index.tolist())
    return summary

In [49]:
def split_and_scale_per_company(df, features=FEATURES, train_frac=0.7, val_frac=0.15):
    out_frames = []
    scalers = {}

    for symbol, group in df.groupby("Symbol"):
        group = group.sort_values("Date").reset_index(drop=True)

        n = len(group)
        train_end = int(n * train_frac)
        val_end = int(n * (train_frac + val_frac))

        group["Split"] = "train"
        group.loc[train_end:val_end, "Split"] = "val"
        group.loc[val_end:, "Split"] = "test"

        scaler = MinMaxScaler()

        # Replace inf values
        group[features] = group[features].replace([np.inf, -np.inf], np.nan)

        # Fill missing values
        group[features] = group[features].ffill().bfill().fillna(0)

        train_rows = group[group["Split"] == "train"]

        if len(train_rows) == 0:
            print(f"Skipping symbol {symbol}")
            continue

        scaler.fit(train_rows[features])
        scalers[symbol] = scaler

        scaled = scaler.transform(group[features])

        for i, feat in enumerate(features):
            group[f"{feat}_scaled"] = scaled[:, i]

        out_frames.append(group)

    return pd.concat(out_frames, ignore_index=True), scalers

In [50]:
def build_sequences(df, window=WINDOW, features=FEATURES, target=TARGET, split="train"):
    scaled_features = [f"{f}_scaled" for f in features]
    target_scaled = f"{target}_scaled"

    X_price, X_symbol, y = [], [], []

    for symbol, group in df.groupby("Symbol"):
        group = group.sort_values("Date").reset_index(drop=True)
        subset = group[group["Split"] == split]
        if len(subset) <= window:
            continue

        arr = subset[scaled_features].values
        target_arr = subset[target_scaled].values
        symbol_id = subset["SymbolID"].iloc[0]

        for i in range(len(arr) - window):
            X_price.append(arr[i:i + window])
            X_symbol.append(symbol_id)
            y.append(target_arr[i + window])

    return (
        np.array(X_price, dtype=np.float32),
        np.array(X_symbol, dtype=np.int32),
        np.array(y, dtype=np.float32),
    )

In [51]:
def build_model(window, n_features, n_symbols, rnn_type="LSTM"):
    RNN = LSTM if rnn_type == "LSTM" else GRU

    price_input = Input(shape=(window, n_features), name="price_seq")
    x = RNN(64, return_sequences=True)(price_input)
    x = Dropout(0.2)(x)
    x = RNN(32)(x)

    symbol_input = Input(shape=(1,), name="symbol_id")
    s = Embedding(input_dim=n_symbols, output_dim=8)(symbol_input)
    s = Flatten()(s)

    combined = Concatenate()([x, s])
    combined = Dense(32, activation="relu")(combined)
    combined = Dropout(0.2)(combined)
    output = Dense(1, name="next_close")(combined)

    model = Model(
        inputs=[price_input, symbol_input],
        outputs=output,
        name=f"nepse_{rnn_type.lower()}"
    )
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    return model

In [52]:
def naive_baseline_mae_returns(df, target="per_change", split="test"):
    """
    Naive baseline for percent-change target: always predict 0% change.
    Uses UNSCALED values so '0' unambiguously means '0% change',
    not wherever 0 happens to land after MinMax scaling.
    """
    errors = []
    for symbol, group in df.groupby("Symbol"):
        subset = group[group["Split"] == split]
        if len(subset) == 0:
            continue
        actual = subset[target].values          # unscaled, raw percent change
        predicted = np.zeros_like(actual)        # naive guess: no movement
        errors.append(np.abs(actual - predicted))
    return np.mean(np.concatenate(errors))

In [53]:
def main(data_folder="data", epochs=30, batch_size=32):
    csv_files = glob.glob(os.path.join(data_folder, "*.csv"))

    if not csv_files:
        raise FileNotFoundError("No CSV files found.")

    # Each CSV is assumed to be one company's history (Symbol inferred from
    # filename inside load_stock_data). Load and stack them all so the
    # per-symbol logic downstream (embeddings, scaling, splits) has
    # multiple symbols to work with.
    all_dfs = [load_stock_data(f) for f in csv_files]
    df = pd.concat(all_dfs, ignore_index=True)
    print("Columns after load:", df.columns.tolist())
    print("Symbols found:", sorted(df['Symbol'].unique()))

    df = load_and_clean(df)

    # Encode categoricals
    df, n_symbols, symbol_to_id = encode_categoricals(df)

    # Sanity-check how much data each symbol has before splitting/scaling
    check_symbol_coverage(df)

    # Per-company scaling + chronological split
    df, scalers = split_and_scale_per_company(df)

    # Build sequences
    Xp_train, Xs_train, y_train = build_sequences(df, split="train")
    Xp_val,   Xs_val,   y_val   = build_sequences(df, split="val")
    Xp_test,  Xs_test,  y_test  = build_sequences(df, split="test")

    print(f"Train: {Xp_train.shape}, Val: {Xp_val.shape}, Test: {Xp_test.shape}")

    # Train LSTM
    lstm_model = build_model(WINDOW, len(FEATURES), n_symbols, rnn_type="LSTM")
    lstm_model.fit(
        [Xp_train, Xs_train], y_train,
        validation_data=([Xp_val, Xs_val], y_val),
        epochs=epochs, batch_size=batch_size
    )

    # Train GRU
    gru_model = build_model(WINDOW, len(FEATURES), n_symbols, rnn_type="GRU")
    gru_model.fit(
        [Xp_train, Xs_train], y_train,
        validation_data=([Xp_val, Xs_val], y_val),
        epochs=epochs, batch_size=batch_size
    )

    # Evaluate both against the naive baseline
    lstm_loss, lstm_mae = lstm_model.evaluate([Xp_test, Xs_test], y_test)
    gru_loss, gru_mae = gru_model.evaluate([Xp_test, Xs_test], y_test)
    baseline_mae = naive_baseline_mae_returns(df, split="test")

    print(f"\nLSTM  test MAE: {lstm_mae:.5f}")
    print(f"GRU   test MAE: {gru_mae:.5f}")
    print(f"Naive baseline MAE (unscaled % change): {baseline_mae:.5f}")

    return {
        "df": df, "scalers": scalers, "symbol_to_id": symbol_to_id,
        "lstm_model": lstm_model, "gru_model": gru_model,
        "lstm_mae": lstm_mae, "gru_mae": gru_mae, "baseline_mae": baseline_mae,
    }


if __name__ == "__main__":
    main()

No symbol column found in data/UNHPL.csv; using filename as Symbol = 'UNHPL'.
No symbol column found in data/NICLBSL.csv; using filename as Symbol = 'NICLBSL'.
No symbol column found in data/SLICLP.csv; using filename as Symbol = 'SLICLP'.
No symbol column found in data/CMF1.csv; using filename as Symbol = 'CMF1'.
No symbol column found in data/NLBBL.csv; using filename as Symbol = 'NLBBL'.
No symbol column found in data/MEROPO.csv; using filename as Symbol = 'MEROPO'.
No symbol column found in data/PIC.csv; using filename as Symbol = 'PIC'.
No symbol column found in data/PFL.csv; using filename as Symbol = 'PFL'.
No symbol column found in data/DORDI.csv; using filename as Symbol = 'DORDI'.
No symbol column found in data/LUK.csv; using filename as Symbol = 'LUK'.
No symbol column found in data/AHL.csv; using filename as Symbol = 'AHL'.
No symbol column found in data/GILBPO.csv; using filename as Symbol = 'GILBPO'.
No symbol column found in data/ALBSLP.csv; using filename as Symbol = 'A